# Verifikasi baseline (C0 lanjutan): versi loss saat training & identitas dataset VisDrone

**V1 — Loss baseline vs loss Ultralytics standar.**
- Commit kode saat baseline dilatih dibaca dari checkpoint (`ckpt["git"]`, ditulis otomatis oleh Ultralytics 8.3.x).
- Repo dikembalikan ke commit itu (git worktree). `model.loss()` lalu dijalankan dengan **bobot baseline yang sebenarnya** pada **batch nyata** dari dataset, di dua lingkungan terpisah: repo@commit dan Ultralytics resmi dengan versi yang sama.
- Yang dibandingkan: loss (box/cls/dfl) dan seluruh gradien. Jalan di CPU, deterministik.
- Vonis IDENTIK berarti selisih relatif ≤ 1e-6, yaitu training baseline secara matematis memakai loss standar.

**V2 — Apakah `/content/visdrone.yaml` (baseline) = `visdrone_person_vehicle.yaml` (M0)?**
- Nama dan jumlah kelas di checkpoint dicocokkan dengan yaml kita.
- **Validasi direproduksi.** `best.pt` dievaluasi ulang di split val yaml kita dengan setelan validasi yang sama seperti saat training (rect, batch = 2 × batch training, FP16, conf 0.001, IoU 0.7). Hasilnya dibandingkan dengan metrik yang tersimpan di checkpoint (`train_metrics`).
- Bila val, label, atau pemetaan kelas berbeda, metriknya tidak akan cocok. Toleransi |Δ| ≤ 0.002 menampung selisih numerik FP16.
- KITTI ikut dijalankan sebagai kontrol metode, karena yaml-nya sudah pasti sama.

**Batas V2.** Identitas split **train** tidak bisa dibuktikan dari checkpoint. V2 hanya membuktikan val dan label sama. Bila isi `/content/visdrone.yaml` masih tersimpan (notebook Colab baseline), tempelkan di sel 1 (`baseline_yaml_text`) untuk dibandingkan langsung.

Prasyarat: GPU T4 (untuk V2), Internet On, dataset `kitti-2class` & `visdrone`, keempat checkpoint baseline. Waktu ±15 menit.

In [ ]:
# [1] KONFIGURASI
import os, sys, json, time, shutil, subprocess, datetime, hashlib, platform
from pathlib import Path
CFG = dict(
    repo_url="https://github.com/maugi1547/thesis-Difficulty-Aware-Router.git",
    repo_dir="/kaggle/working/thesis-Difficulty-Aware-Router",
    code_commit="5416543",
    out_dir="/kaggle/working/verifikasi_baseline",
    baselines={
        "Vanilla KITTI": dict(ckpt="/kaggle/input/models/agungmaugi/yolov8n-kitti/pytorch/default/1/Skenario1_YOLOv8n_Standar/weights/best.pt",
                              data_yaml="/kaggle/input/datasets/agungmaugi/kitti-2class/kitti_2class.yaml"),
        "P2-Static KITTI": dict(ckpt="/kaggle/input/models/agungmaugi/yolov8n-p2-statis-kitti/pytorch/default/1/Skenario2_YOLOv8n_P2_Statis/weights/best.pt",
                                data_yaml="/kaggle/input/datasets/agungmaugi/kitti-2class/kitti_2class.yaml"),
        "Vanilla VisDrone": dict(ckpt="/kaggle/input/ISI_PATH_VANILLA_VISDRONE/weights/best.pt",
                                 data_yaml="/kaggle/input/datasets/agungmaugi/visdrone/datasets/VisDrone_1/visdrone_person_vehicle.yaml"),
        "P2-Static VisDrone": dict(ckpt="/kaggle/input/ISI_PATH_P2STATIC_VISDRONE/weights/best.pt",
                                   data_yaml="/kaggle/input/datasets/agungmaugi/visdrone/datasets/VisDrone_1/visdrone_person_vehicle.yaml"),
    },
    tol_metric=0.002, loss_batch_images=8,
    baseline_yaml_text="",          # opsional: tempel isi /content/visdrone.yaml milik baseline di sini
)
CFG.update(globals().get("CFG_OVERRIDE", {}))
assert "ISI_" not in json.dumps(CFG) or "CFG_OVERRIDE" in globals(), "Isi path checkpoint VisDrone (ISI_...) di sel 1."
REPO = Path(CFG["repo_dir"]); OUT = Path(CFG["out_dir"]); OUT.mkdir(parents=True, exist_ok=True)
PROV = {"started_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"), "cfg": CFG}

def sh(cmd, cwd=None, secret=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if secret:
        out = out.replace(secret, "***")
    if r.returncode != 0:
        raise RuntimeError(f"{' '.join(c if c != secret else '***' for c in cmd)}\n{out}")
    return out

In [ ]:
# [2] CLONE dari GitHub + checkout commit terkunci (clone baru setiap sesi -> tidak ada sisa state lama)
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass                                              # repo publik: token tidak perlu
url = CFG["repo_url"]
auth_url = url.replace("https://", f"https://x-access-token:{token}@") if token else url

if REPO.exists():
    shutil.rmtree(REPO)
sh(["git", "clone", "--quiet", auth_url, str(REPO)], secret=token)
sh(["git", "remote", "set-url", "origin", url], cwd=REPO)          # token tidak tersimpan di .git/config
try:
    sh(["git", "checkout", "--quiet", CFG["code_commit"]], cwd=REPO)
except RuntimeError as e:
    raise RuntimeError(f"Commit {CFG['code_commit']} tidak ada di GitHub. Sudah di-push? "
                       f"HEAD remote = {sh(['git', 'log', '-1', '--format=%h %s', 'origin/HEAD'], cwd=REPO)}") from e
PROV["git_commit"] = sh(["git", "rev-parse", "HEAD"], cwd=REPO)
PROV["git_commit_msg"] = sh(["git", "log", "-1", "--format=%s (%cd)"], cwd=REPO)
PROV["git_branches_containing"] = sh(["git", "branch", "-r", "--contains", "HEAD"], cwd=REPO).split()
print("✅ commit:", PROV["git_commit"][:10], "|", PROV["git_commit_msg"], "| branch:", PROV["git_branches_containing"],
      "| token:", "ya" if token else "tidak")

In [ ]:
# [3] INSTALL editable dari repo (ultralytics pip bawaan dihapus agar tidak tertukar)
print(sh([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "ultralytics"]) or "uninstall: ok")
print(sh([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)])[-500:] or "install: ok")

In [ ]:
# [6] PATCH SUMBER (idempotent, EOL-aware) — hapus sel ini setelah kedua patch di-commit ke repo
#   P2   block.py : simpan keputusan & logit gate mentah per-gambar di forward eval (uji 0.5 + log validasi).
#                   Di dalam blok `not torch.jit.is_tracing()` -> graf export tidak berubah.
#   CONF loss.py  : difficulty score = entropy + var + conf (tanda minus lama = salah ketik, 2026-10-09).
sys.path.insert(0, str(REPO))
import router_train_utils as U
for P_ in (U.PATCH_P2, U.PATCH_CONF_SIGN):
    print(U.apply_source_patch(REPO / P_["rel"], P_["old"], P_["new"], P_["done_marker"], P_["name"]))
PROV["git_diff_sha256"] = hashlib.sha256(sh(["git", "diff"], cwd=REPO).encode()).hexdigest()
PROV["git_status"] = sh(["git", "status", "--short"], cwd=REPO)
print("git status:\n" + PROV["git_status"])

In [ ]:
# [7] VERIFIKASI VERSI KODE — berhenti di sini bila sesi ini tidak memakai kode yang dimaksud
import inspect
sys.path.insert(0, str(REPO)); os.chdir(REPO)
import torch, ultralytics
import ultralytics.nn.modules.block as _blk
import ultralytics.model_yolov8_router as _mr
import ultralytics.utils.loss as _ls
_src_r = inspect.getsource(_blk.UltraLightWeightDifficultyAwareRouter)
CODE_FEATURES = {
    "ultralytics di-import dari repo (bukan pip)": Path(ultralytics.__file__).resolve().is_relative_to(REPO.resolve()),
    "AUDIT-FIX #9 kalibrasi = buffer int64": "_calib_T_micro" in _src_r,
    "GroupNorm di classifier gate": "GroupNorm(1, hidden_dim // 2)" in _src_r,
    "top-k spatial pool di gate": "_topk_spatial_pool" in _src_r,
    "patch P2 keputusan + logit gate per-gambar": "_last_gate_logit_raw" in _src_r,
    "fix tanda conf (+) di difficulty score": "entropy_f + var_f + conf_f" in inspect.getsource(_ls.v8DetectionLoss),
    "AUDIT-FIX #6 kolom GV tanpa lag": hasattr(_mr.DualBranchDetectionLoss, "_flush_pending_log"),
    "AUDIT-FIX #3 advantage di-clip": "gate_adv_clip" in inspect.getsource(_mr.DualBranchDetectionLoss),
    "AUDIT-FIX #7 normalisasi bg anchor": "normalize_bg_anchors" in inspect.getsource(_ls.v8DetectionLoss),
}
for k, v in CODE_FEATURES.items():
    print(("✅ " if v else "❌ ") + k)
PROV.update(torch=torch.__version__, ultralytics=ultralytics.__version__, python=platform.python_version(),
            cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
            code_features=CODE_FEATURES)
print({k: PROV[k] for k in ("torch", "ultralytics", "cuda", "gpu")})
assert all(CODE_FEATURES.values()), "Versi kode BUKAN yang dimaksud — push/clone ulang, lalu Restart & Run All."
assert torch.cuda.is_available() or "CFG_OVERRIDE" in globals(), "GPU tidak aktif (Settings → Accelerator → GPU T4)."

In [ ]:
%%writefile /kaggle/working/cek_loss_env.py
"""Jalankan model.loss() di SATU lingkungan ultralytics (path diberikan) pada bobot & batch yang sama; simpan loss + gradien."""
import sys, json
env, wpath, bpath, out = sys.argv[1:5]
sys.path.insert(0, env)
import torch
import ultralytics
from ultralytics.nn.tasks import DetectionModel
from ultralytics.cfg import get_cfg
from ultralytics.utils import DEFAULT_CFG
assert ultralytics.__file__.startswith(env), f"ultralytics diimpor dari {ultralytics.__file__}, bukan {env}"
torch.manual_seed(0); torch.use_deterministic_algorithms(True)
W = torch.load(wpath, map_location="cpu", weights_only=False)
B = torch.load(bpath, map_location="cpu", weights_only=False)
m = DetectionModel(cfg=W["yaml"], ch=3, nc=W["nc"], verbose=False)
miss, unexp = m.load_state_dict(W["sd"], strict=False)
assert not [k for k in miss] and not unexp, (miss[:5], unexp[:5])
m.args = get_cfg(DEFAULT_CFG, overrides=W["hyp"])
m.float().train()
batch = {"img": B["img"].float() / 255.0, "cls": B["cls"], "bboxes": B["bboxes"], "batch_idx": B["batch_idx"]}
res = m.loss(batch)
total, items = res[0], res[1]
total = total.sum() if total.dim() else total
total.backward()
grads = torch.cat([p.grad.flatten() for p in m.parameters() if p.grad is not None])
torch.save({"total": float(total), "items": items.detach().float().flatten()[:3], "n_items": int(items.numel()),
            "grads": grads, "file": ultralytics.__file__, "version": ultralytics.__version__}, out)
print(json.dumps({"env": env, "version": ultralytics.__version__, "total": float(total),
                  "items": [round(float(x), 6) for x in items.flatten()]}))

In [ ]:
%%writefile /kaggle/working/cek_loss_compare.py
"""Bandingkan loss + gradien dua lingkungan (A = referensi upstream, B = versi repo)."""
import sys, json, subprocess, torch
envA, envB, wpath, bpath, tag = sys.argv[1:6]
outs = []
for nm, env in (("upstream", envA), ("repo", envB)):
    o = f"/tmp/cek_loss_{tag}_{nm}.pt"
    r = subprocess.run([sys.executable, __file__.replace("compare", "env"), env, wpath, bpath, o], capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-2000:], r.stderr[-3000:]); raise SystemExit(f"gagal di {nm}")
    print(r.stdout.strip().splitlines()[-1]); outs.append(torch.load(o, weights_only=False))
a, b = outs
d_items = (a["items"] - b["items"]).abs().max().item()
d_total = abs(a["total"] - b["total"]) / max(abs(a["total"]), 1e-12)
same_shape = a["grads"].shape == b["grads"].shape
d_grad = ((a["grads"] - b["grads"]).abs().max() / a["grads"].abs().max()).item() if same_shape else float("nan")
verdict = "IDENTIK" if (d_items < 1e-6 and d_total < 1e-6 and d_grad < 1e-6) else "BERBEDA"
print(json.dumps(dict(tag=tag, d_items_maks=d_items, d_total_rel=d_total, d_grad_rel_maks=d_grad,
                      n_items_upstream=a["n_items"], n_items_repo=b["n_items"], vonis=verdict)))

In [ ]:
# [8] METADATA CHECKPOINT: commit, versi, tanggal, kelas, metrik val tersimpan
import torch, yaml, pandas as pd, numpy as np
META = {}
for nm, b in CFG["baselines"].items():
    ck = torch.load(b["ckpt"], map_location="cpu", weights_only=False)
    mdl = ck.get("ema") or ck.get("model")
    META[nm] = dict(date=ck.get("date"), version=ck.get("version"), git=ck.get("git"),
                    names=dict(getattr(mdl, "names", {}) or {}), nc=int(mdl.yaml.get("nc", -1)) if hasattr(mdl, "yaml") else None,
                    data=ck["train_args"].get("data"), batch=ck["train_args"].get("batch"), epoch=ck.get("epoch"),
                    train_metrics=ck.get("train_metrics"))
    print(f"\n== {nm}\n  tanggal {META[nm]['date']} | versi {META[nm]['version']} | git {META[nm]['git']}\n"
          f"  data {META[nm]['data']} | kelas {META[nm]['names']} | batch {META[nm]['batch']}\n  train_metrics {META[nm]['train_metrics']}")
(OUT / "metadata_baseline.json").write_text(json.dumps(META, indent=1, default=str))

In [ ]:
# [9] V1 — LOSS saat training baseline vs Ultralytics resmi (versi sama), bobot baseline asli, batch nyata, CPU
from ultralytics.data import YOLODataset
from ultralytics.data.utils import check_det_dataset
FULL = Path("/kaggle/working/repo_full")
if not FULL.exists():
    sh(["git", "clone", "--quiet", CFG["repo_url"], str(FULL)])
    sh(["git", "fetch", "--quiet", "--all"], cwd=FULL)
V1 = {}
for nm, b in CFG["baselines"].items():
    g = META[nm]["git"] or {}
    commit, ver = g.get("commit"), META[nm]["version"]
    if not commit:
        V1[nm] = dict(vonis="TIDAK DAPAT DIUJI", alasan=f"checkpoint tidak menyimpan commit git (git={g})"); print(nm, V1[nm]); continue
    wt = Path(f"/kaggle/working/wt_{commit[:8]}")
    if not wt.exists():
        try:
            sh(["git", "worktree", "add", "--detach", str(wt), commit], cwd=FULL)
        except RuntimeError as e:
            V1[nm] = dict(vonis="TIDAK DAPAT DIUJI", alasan=f"commit {commit} tidak ada di GitHub (belum di-push?)"); print(nm, V1[nm]); continue
    up = Path(f"/kaggle/working/ultra_{ver}")
    if not up.exists():
        sh([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--target", str(up), f"ultralytics=={ver}"])
    ck = torch.load(b["ckpt"], map_location="cpu", weights_only=False)
    mdl = (ck.get("ema") or ck["model"]).float()
    wpath = OUT / f"w_{nm.replace(' ', '_')}.pt"
    torch.save({"yaml": mdl.yaml, "nc": int(mdl.yaml["nc"]), "sd": mdl.state_dict(),
                "hyp": {k: ck["train_args"][k] for k in ("box", "cls", "dfl")}}, wpath)
    D = check_det_dataset(b["data_yaml"])
    ds = YOLODataset(img_path=D["train"], data=D, task="detect", imgsz=640, augment=False, rect=False, batch_size=8)
    bt = ds.collate_fn([ds[i] for i in range(CFG["loss_batch_images"])])
    bpath = OUT / f"batch_{nm.split()[-1]}.pt"
    torch.save({k: bt[k] for k in ("img", "cls", "bboxes", "batch_idx")}, bpath)
    r = subprocess.run([sys.executable, "/kaggle/working/cek_loss_compare.py", str(up), str(wt), str(wpath), str(bpath), nm.replace(" ", "_")],
                       capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:] if r.returncode else "")
    V1[nm] = {**json.loads(r.stdout.strip().splitlines()[-1]), "commit": commit, "version": ver} if r.returncode == 0 else dict(vonis="GAGAL", log=r.stderr[-800:])
(OUT / "v1_loss.json").write_text(json.dumps(V1, indent=1, default=str))
print(pd.DataFrame(V1).T.to_string())

In [ ]:
# [10] V2 — KELAS + REPRODUKSI METRIK VAL (setelan validasi saat training: rect, batch = 2 x batch, FP16)
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

def _count_images(data_yaml, split):
    p = check_det_dataset(data_yaml)[split]
    ps = p if isinstance(p, list) else [p]
    n = 0
    for q in map(Path, ps):
        n += sum(1 for _ in q.rglob("*") if _.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")) if q.is_dir() else len(q.read_text().split())
    return n

V2 = {}
for nm, b in CFG["baselines"].items():
    D = yaml.safe_load(Path(b["data_yaml"]).read_text())
    names_yaml = D["names"] if isinstance(D["names"], dict) else dict(enumerate(D["names"]))
    names_ck = {int(k): v for k, v in META[nm]["names"].items()}
    m = YOLO(b["ckpt"])
    r = m.val(data=b["data_yaml"], split="val", imgsz=640, batch=2 * int(META[nm]["batch"]), half=True, conf=0.001, iou=0.7,
              rect=True, plots=False, verbose=False, workers=2, project=str(OUT / "val_runs"), name=nm.replace(" ", "_"))
    now = {"metrics/precision(B)": r.box.mp, "metrics/recall(B)": r.box.mr, "metrics/mAP50(B)": r.box.map50, "metrics/mAP50-95(B)": r.box.map}
    tm = META[nm]["train_metrics"] or {}
    d = {k: (float(now[k]) - float(tm[k])) if k in tm else None for k in now}
    ok_m = all(v is not None and abs(v) <= CFG["tol_metric"] for v in d.values())
    V2[nm] = dict(kelas_ckpt=names_ck, kelas_yaml={int(k): v for k, v in names_yaml.items()},
                  kelas_sama=[str(names_ck[k]).lower() for k in sorted(names_ck)] == [str(names_yaml[k]).lower() for k in sorted(names_yaml)],
                  metrik_sekarang={k: round(float(v), 5) for k, v in now.items()},
                  metrik_tersimpan={k: round(float(tm[k]), 5) for k in now if k in tm},
                  selisih={k: (round(v, 5) if v is not None else None) for k, v in d.items()},
                  n_val=_count_images(b["data_yaml"], "val"),
                  vonis="COCOK" if ok_m else "TIDAK COCOK")
    print(f"\n== {nm}: {V2[nm]['vonis']} | kelas sama: {V2[nm]['kelas_sama']} | n_val {V2[nm]['n_val']}\n  selisih {V2[nm]['selisih']}")
if CFG["baseline_yaml_text"].strip():
    by = yaml.safe_load(CFG["baseline_yaml_text"]); oy = yaml.safe_load(Path(CFG["baselines"]["Vanilla VisDrone"]["data_yaml"]).read_text())
    V2["_yaml_langsung"] = {k: dict(baseline=by.get(k), m0=oy.get(k)) for k in sorted(set(by) | set(oy))}
    print("\nPerbandingan yaml langsung:\n" + json.dumps(V2["_yaml_langsung"], indent=1))
(OUT / "v2_dataset.json").write_text(json.dumps(V2, indent=1, default=str))

In [ ]:
# [11] RINGKASAN + ARSIP
import zipfile
print("V1 (loss):", {k: v.get("vonis") for k, v in V1.items()})
print("V2 (val & kelas):", {k: (v.get("vonis"), v.get("kelas_sama")) for k, v in V2.items() if not k.startswith("_")})
arc = Path("/kaggle/working/verifikasi_baseline.zip")
with zipfile.ZipFile(arc, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*.json"):
        z.write(f, f.relative_to(OUT))
print("📦", arc)